In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_parquet("s3://sliderule-public/atl24r3/bluetopo/ATL24_20200531160408_10100702_007_01_003_01_spot3.parquet")

In [ ]:
df["bluetopo_h"] = -(df["bluetopo_depth"] + df["bluetopo_correction"])

In [ ]:

bathy = df.dropna(subset=["bathy_ortho_h"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(bathy["x_atc"], bathy["bathy_ortho_h"], s=2)
ax.scatter(bathy["x_atc"], bathy["bluetopo_h"], s=2)
ax.set_xlim(3.8e6, 4.4e6)
ax.set_ylim(-200, 50)
ax.set_xlabel("x_atc (m)")
ax.set_ylabel("height (m)")
ax.set_title(f"Mean bathymetry orthometric height per bin ({len(bathy)} of {len(df)} bins)")
ax.grid(True)
plt.show()

In [ ]:
bathy = df.dropna(subset=["kd"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(bathy["x_atc"], bathy["kd"], s=2)
ax.set_xlim(3.8e6, 4.4e6)
ax.set_xlabel("x_atc (m)")
ax.grid(True)
plt.show()

In [ ]:
import boto3, geopandas as gpd
from botocore import UNSIGNED
from botocore.config import Config
from shapely.geometry import box

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
bucket = "noaa-ocs-nationalbathymetry-pds"
resp = s3.list_objects_v2(Bucket=bucket, Prefix="BlueTopo/_BlueTopo_Tile_Scheme/")
key = max((o for o in resp["Contents"] if o["Key"].endswith(".gpkg")),
          key=lambda o: o["Key"])["Key"]
s3.download_file(bucket, key, "tile_scheme.gpkg")

tiles = gpd.read_file("tile_scheme.gpkg").to_crs(4326)

# Keep only tiles that actually have data (check the column names; I believe
# it's GeoTIFF_Link, but print tiles.columns to confirm)
tiles = tiles[tiles["GeoTIFF_Link"].notna()]

# Restrict to CONUS so Alaska, Hawaii, PR and Guam don't stretch the hull
conus = box(-130, 22, -60, 50)
tiles = tiles[tiles.intersects(conus)]

hull = tiles.union_all().convex_hull   # use unary_union on older geopandas
print(hull.wkt)
s = gpd.GeoSeries([hull], crs=4326)
s.to_file("bluetopo_conus_hull.geojson", driver="GeoJSON")

In [ ]:
"""Read bluetopo_conus_hull.geojson and plot it over a world map.

Requires: pip install geopandas matplotlib cartopy
"""
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

hull = gpd.read_file("bluetopo_conus_hull.geojson").to_crs(4326)

fig = plt.figure(figsize=(14, 7))

# --- Full world view ---
ax = fig.add_subplot(1, 2, 1, projection=ccrs.Robinson())
ax.set_global()
ax.add_feature(cfeature.OCEAN, facecolor="#dceaf5")
ax.add_feature(cfeature.LAND, facecolor="#f0ede4")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
ax.add_geometries(hull.geometry, crs=ccrs.PlateCarree(),
                  facecolor="red", edgecolor="darkred", alpha=0.5, linewidth=1.2)
ax.set_title("World view")

# --- Zoomed to the hull ---
minx, miny, maxx, maxy = hull.total_bounds
pad = 5
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
ax2.set_extent([minx - pad, maxx + pad, miny - pad, maxy + pad], crs=ccrs.PlateCarree())
ax2.add_feature(cfeature.OCEAN, facecolor="#dceaf5")
ax2.add_feature(cfeature.LAND, facecolor="#f0ede4")
ax2.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax2.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":")
ax2.add_feature(cfeature.STATES, linewidth=0.2)
ax2.add_geometries(hull.geometry, crs=ccrs.PlateCarree(),
                   facecolor="red", edgecolor="darkred", alpha=0.4, linewidth=1.5)
ax2.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
ax2.set_title("Zoomed to BlueTopo CONUS hull")

plt.tight_layout()
plt.savefig("bluetopo_conus_hull.png", dpi=150)
plt.show()

In [ ]:
from shapely.geometry.polygon import orient
from sliderule import sliderule, earthdata

sliderule.init("localhost", organization=None, verbose=True)

# SlideRule expects a closed, counter-clockwise ring of {"lon", "lat"} points
poly_geom = orient(s.to_crs(4326).union_all(), sign=1.0)
poly = [{"lon": x, "lat": y} for x, y in poly_geom.exterior.coords]

parms = {
    "asset": "icesat2",          # ATL03
    "poly": poly,
    "max_resources": 100000,     # CONUS-scale region; the default limit is far too small
}
granules = earthdata.search(parms)
print(f"{len(granules)} ATL03 granules intersect the polygon")
granules[:10]

In [ ]:
bluetopo_granules = granules
bluetopo_infos = {granule[6:29] for granule in bluetopo_granules}
with open("/data/ATL24/atl03_bluetopo_granules.txt", "w") as file:
    for granule in granules:
        file.write(f"{granule.replace("ATL03", "ATL24").replace(".h5", "_003_01.parquet")}\n")

In [ ]:
atl24_granules = [granule.strip() for granule in open("/data/ATL24/atl24r3_granules.txt", "r") if granule.strip()]
selected_granules = [granule for granule in atl24_granules if granule[6:29] in bluetopo_infos]
print(len(selected_granules))

In [ ]:
with open("/data/ATL24/atl24r3_bluetopo_granules.txt", "w") as file:
    for granule in selected_granules:
        file.write(f"{granule.replace(".h5", ".parquet")}\n")